# RAG01｜《西遊記》悟空成長檔案館：建立向量資料庫

**資料範圍：第一至七回，自行整理的章節摘要，並非小說原文。**

執行順序：安裝套件 → 上傳本專案的摘要 ZIP → 擷取正文 → EmbeddingGemma 2 → FAISS → 匯出 `rag-data.zip`。

註：沿用課堂的 EmbeddingGemma 2、Sentence Transformers、FAISS 與索引格式。

## 1. 安裝課堂指定套件

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 讀入章節摘要 ZIP（不要在這一步使用 rag-data.zip）

In [ ]:
import io, os, json, re, hashlib, tempfile, zipfile
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import torch, faiss
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi
from google.colab import files

uploaded = files.upload()   # 選「西遊記_第一至七回_章節摘要.zip」
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('請一次上傳一個章節摘要 ZIP')
SOURCE_NAME = zip_names[0]
SOURCE_SHA256 = hashlib.sha256(uploaded[SOURCE_NAME]).hexdigest()
with zipfile.ZipFile(io.BytesIO(uploaded[SOURCE_NAME])) as z:
    texts = [(n, z.read(n).decode('utf-8-sig')) for n in sorted(z.namelist()) if n.lower().endswith('.txt') and '/' not in n]
if not texts:
    raise ValueError('找不到 TXT 章節摘要')
print('讀入章節：',len(texts),[x[0] for x in texts])

## 3. 自訂文字清理及 metadata 擷取

只將每回的「標題＋【內容】」送入向量模型；把章節編號、來源存於 metadata，避免讓作業提醒混入檢索。

In [ ]:
def extract_rag_text(raw):
    title = re.search(r'^標題[：:]\s*(.+)$',raw,re.M)
    body = re.search(r'^【內容】\s*\n(.*?)(?=^【[^】]+】|\Z)',raw,re.M|re.S)
    if not body or not body.group(1).strip():
        raise ValueError('找不到【內容】區塊')
    return (title.group(1).strip()+'\n\n' if title else '')+body.group(1).strip()

def extract_metadata(raw, filename):
    def field(k):
        m=re.search(r'^'+re.escape(k)+r'[：:]\s*(.+)$',raw,re.M)
        return m.group(1).strip() if m else None
    return {'title':field('標題') or filename,'source_url':field('來源網址'),
            'chapter':field('章節'),'filename':filename,
            'document_type':field('資料類型')}

documents=[{'doc_id':f'doc_{i:03d}', 'text':extract_rag_text(raw),
            'metadata':extract_metadata(raw,name)} for i,(name,raw) in enumerate(texts,1)]
index_doc_ids=[d['doc_id'] for d in documents]
for d in documents: print(d['doc_id'],d['metadata']['title'],len(d['text']))

## 4. 使用 EmbeddingGemma 2 建立向量

若模型下載失敗，確認網路與套件版本；建議 GPU 執行。

In [ ]:
MODEL_ID='google/embeddinggemma-2'
MODEL_REVISION=HfApi().model_info(MODEL_ID,token=False).sha
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DTYPE=torch.bfloat16 if DEVICE=='cuda' and torch.cuda.is_bf16_supported() else torch.float32
model=SentenceTransformer(MODEL_ID, revision=MODEL_REVISION,token=False,
    device=DEVICE,config_kwargs={'vision_config':None,'audio_config':None},
    model_kwargs={'dtype':DTYPE,'attn_implementation':'eager'})
DOCUMENT_TEMPLATE='title: none | text: {text}'
QUERY_TEMPLATE='task: search result | query: {question}'
TOKEN_LIMIT=2048
model.max_seq_length=TOKEN_LIMIT
inputs=[DOCUMENT_TEMPLATE.format(text=d['text']) for d in documents]
for item in inputs:
    if len(model.tokenizer(item,add_special_tokens=True,truncation=False)['input_ids'])>TOKEN_LIMIT:
        raise ValueError('文字過長；請切分章節後再試')
vectors=model.encode(inputs,prompt='',batch_size=4,convert_to_numpy=True,
                     normalize_embeddings=True,show_progress_bar=True)
vectors=np.ascontiguousarray(vectors,dtype=np.float32)
faiss.normalize_L2(vectors)
DIMENSION=vectors.shape[1]
index=faiss.IndexFlatIP(DIMENSION)
index.add(vectors)
print('文件：',index.ntotal,'維度：',DIMENSION,'模型版本：',MODEL_REVISION)

## 5. 測試檢索結果

先觀察相似文件是不是有回答到問題，再接 LLM。

In [ ]:
question='孫悟空為什麼自稱齊天大聖？'
q=model.encode([QUERY_TEMPLATE.format(question=question)],prompt='',convert_to_numpy=True,normalize_embeddings=True)
q=np.ascontiguousarray(q,dtype=np.float32);faiss.normalize_L2(q)
scores,inds=index.search(q,3)
for j,score in zip(inds[0],scores[0]):
    print(index_doc_ids[int(j)],f'相似度 {score:.3f}',documents[int(j)]['metadata']['title'])

## 6. 匯出作業要求的 `rag-data.zip`

ZIP 內含索引、文件、對照與模型設定；**不要將 API Key 放入 ZIP**。RAG02 會使用同一版本的 embedding 模型。

In [ ]:
EXPORT=Path(tempfile.mkdtemp(prefix='wukong_rag_'))
faiss.write_index(index,str(EXPORT/'index.faiss'))
(EXPORT/'documents.json').write_text(json.dumps(documents,ensure_ascii=False,indent=2),encoding='utf-8')
(EXPORT/'index_doc_ids.json').write_text(json.dumps(index_doc_ids,ensure_ascii=False,indent=2),encoding='utf-8')
config={'format_version':'1.0','model_id':MODEL_ID,'model_revision':MODEL_REVISION,
  'model_config_kwargs':{'vision_config':None,'audio_config':None},'dimension':int(DIMENSION),
  'normalize_embeddings':True,'index_type':'IndexFlatIP',
  'metric':'cosine_similarity_via_normalized_inner_product',
  'document_template':DOCUMENT_TEMPLATE,'query_template':QUERY_TEMPLATE,'encode_prompt':'',
  'max_seq_length':TOKEN_LIMIT,'document_count':len(documents),
  'document_unit':'one_document_one_vector',
  'file_sha256':{name:hashlib.sha256((EXPORT/name).read_bytes()).hexdigest()
                 for name in ('index.faiss','documents.json','index_doc_ids.json')},
  'source_archive':SOURCE_NAME,'source_archive_sha256':SOURCE_SHA256}
(EXPORT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
OUTPUT_ZIP=Path('/content/rag-data.zip')
with zipfile.ZipFile(OUTPUT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('index.faiss','documents.json','index_doc_ids.json','config.json'):
        z.write(EXPORT/name,arcname=name)
with zipfile.ZipFile(OUTPUT_ZIP) as z:
    assert z.testzip() is None
print('成功產生：',OUTPUT_ZIP)
files.download(str(OUTPUT_ZIP))